# Experiment 1 — Random-Effects Sensitivity Analysis

**Purpose:** Resolve the fixed-effects generalizability gap by re-running the primary Experiment 1 analyses
with `model` (LLM identity) treated as a random rather than fixed effect. This allows population-level
inference across the space of LLM architectures, rather than inferences limited to the six specific
models tested.

**Original specification (exp_1_ana4.ipynb):** All Experiment 1 models include `model` as a fixed
categorical predictor (5 dummy-coded contrasts relative to Gemma3:12b). This restricts generalisation
to these six architectures.

**Sensitivity specification (this notebook):** `model` is removed from fixed effects and added as a
random intercept `(1 | model)`. With only six levels, the between-model variance estimate carries
substantial uncertainty, but the approach (a) allows population-level inference for the remaining
fixed effects and (b) partitions model-identity variance from the residual, providing an ICC for
model-level clustering.

**Models re-fitted:**
1. Reading Hallucination GLMM (perceived trials only, binomial)
2. Recognition Accuracy GLMM (perceived trials only, binomial)
3. Relatedness Rating LMM (all trials, Gaussian)

**Key question:** Do the primary fixed-effect inferences (memory format, source, reading
hallucination effects) replicate when model identity is treated as a random effect?

---
## 0. Setup

In [ ]:
import warnings
import gc
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display, Markdown

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

from pymer4.models import lmer, glmer, compare

import rmllm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
np.random.seed(42)

_lme4   = importr('lme4')
_base_r = importr('base')

data_dir = rmllm.config.PROCESSED_DATA_DIR

# Optimiser controls
_BOBYQA_G    = "glmerControl(optimizer='bobyqa', optCtrl=list(maxfun=500000))"
_BOBYQA_L    =  "lmerControl(optimizer='bobyqa', optCtrl=list(maxfun=500000))"
_NLOPTWRAP_G = "glmerControl(optimizer='nloptwrap', optCtrl=list(maxfun=500000))"
_NLOPTWRAP_L =  "lmerControl(optimizer='nloptwrap', optCtrl=list(maxfun=500000))"

print('Environment ready.')

---
## 1. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(data_dir / 'exp1_trial_data.csv')

# ── Derived variables ─────────────────────────────────────────────────────────
df['read_hallucination'] = 1 - df['trial_compliance']
df['order_c']            = df['order'].astype(int) - 1          # 1/2 → 0/1
df['rating_cen']         = df['rating'] - df['rating'].mean()
df['confidence_num']     = pd.to_numeric(df['confidence'], errors='coerce')

# ── Factor columns ────────────────────────────────────────────────────────────
for col in ['model', 'memory', 'source']:
    df[col] = df[col].astype(str)

# ── Perceived-only subset ────────────────────────────────────────────────────
df_per = df[df['source'] == 'perceived'].copy()

print(f'All trials : {len(df):,}  (models: {df["model"].nunique()}, '
      f'memory: {list(df["memory"].unique())})')
print(f'Perceived  : {len(df_per):,}')
print(f'\nModel levels (n={df["model"].nunique()}): {sorted(df["model"].unique())}')
display(df[['read_hallucination','accuracy','rating_cen','confidence_num',
            'order_c']].describe().round(3))

---
## 2. Helper Functions

In [ ]:
def _to_polars(data):
    pf = pl.from_pandas(data) if isinstance(data, pd.DataFrame) else data
    cat_cols = ['model', 'memory', 'source']
    casts = {c: pl.String for c in cat_cols if c in pf.columns}
    return pf.cast(casts) if casts else pf


def _setup_factors(m, ref_levels=None):
    """Set factor reference levels via pymer4.set_factors."""
    cols = m.data.columns if hasattr(m.data, 'columns') else []
    factors = {}
    if 'memory' in cols:
        factors['memory'] = ['SingleTurn', 'TrialChain']
    if 'source' in cols:
        factors['source'] = ['perceived', 'imagined']
    if 'model' in cols:
        factors['model'] = sorted(m.data['model'].unique().to_list()
                                  if hasattr(m.data['model'], 'unique')
                                  else [])
    if factors and ref_levels is None:
        m.set_factors(factors)


def is_singular(m):
    try:
        return bool(_lme4.isSingular(m.r_model)[0])
    except Exception:
        return False


def _get_aic(m):
    try:
        return float(m.result_fit_stats['AIC'][0])
    except Exception:
        try:
            return float(ro.r('AIC')(m.r_model)[0])
        except Exception:
            return np.nan


def _get_loglik(m):
    try:
        return float(ro.r('logLik')(m.r_model)[0])
    except Exception:
        return np.nan


def show_fe(m, label='', exponentiate=False):
    tbl = m.result_fit.to_pandas() if isinstance(m.result_fit, pl.DataFrame) else m.result_fit
    p_col = next((c for c in tbl.columns
                  if c.lower() in ('p_value', 'p', 'pr(>|z|)', 'pr(>|t|)')), None)
    if p_col:
        tbl['sig'] = tbl[p_col].map(
            lambda p: '***' if pd.notnull(p) and p < .001
                      else '**'  if pd.notnull(p) and p < .01
                      else '*'   if pd.notnull(p) and p < .05
                      else '')
    if exponentiate:
        for raw, name in [('estimate', 'OR'), ('Estimate', 'OR'),
                          ('conf_low', 'OR_lo'), ('conf_high', 'OR_hi')]:
            if raw in tbl.columns:
                tbl[name] = np.exp(tbl[raw])
    display(Markdown(f'**{label} — Fixed effects**'))
    display(tbl.round(4))
    return tbl


def extract_var_components(m, label=''):
    """Extract variance components (VarCorr) from a fitted lme4 model.

    Note: R identifiers cannot start with underscore; use 'tmpmod', not '_tmp_model'.
    For binomial GLMMs (logit link) the residual variance is not estimated by lme4;
    the latent-variable ICC uses sigma^2_residual = pi^2/3 ≈ 3.29.
    """
    display(Markdown(f'**{label} — Variance components**'))
    r_mod = getattr(m, 'r_model', None)
    if r_mod is None:
        print('  r_model not available.')
        return None
    try:
        ro.globalenv['tmpmod'] = r_mod
        vc_r = ro.r('as.data.frame(VarCorr(tmpmod))')
        with localconverter(ro.default_converter + pandas2ri.converter):
            vc = ro.conversion.rpy2py(vc_r)
    except Exception as e:
        print(f'  [VarCorr failed: {e}]')
        return None
    display(vc.round(6))

    # Identify family (Gaussian has Residual row; binomial does not)
    has_residual = 'Residual' in vc['grp'].values
    model_var    = float(vc[vc['grp'] == 'model']['vcov'].values[0]) \
                   if 'model' in vc['grp'].values else 0.0
    total_random = float(vc['vcov'].sum())  # sum of random effects variances only

    if has_residual:
        # Gaussian LMM: ICC = sigma^2_model / (sigma^2_model + sigma^2_residual)
        resid_var  = float(vc[vc['grp'] == 'Residual']['vcov'].values[0])
        total_var  = total_random  # already includes residual
        icc_model  = model_var / total_var if total_var > 0 else np.nan
        print(f'  Model ICC (Gaussian)  = {icc_model:.4f}  '
              f'(σ²_model={model_var:.4f}, σ²_total={total_var:.4f})')
    else:
        # Binomial GLMM: latent-variable ICC uses pi^2/3 for residual
        pi2_3     = (np.pi ** 2) / 3   # ≈ 3.2899
        total_var  = total_random + pi2_3
        icc_model  = model_var / total_var if total_var > 0 else np.nan
        print(f'  Model ICC (latent, logit) = {icc_model:.4f}  '
              f'(σ²_model={model_var:.4f}, σ²_resid[latent]={pi2_3:.4f}, '
              f'σ²_total[latent]={total_var:.4f})')

    return {'model_var': model_var, 'total_var': total_var,
            'icc_model': icc_model, 'has_residual': has_residual}


def fit_glmm(formula, data, label='', fallback_formula=None):
    """Fit a GLMM (binomial) via pymer4 glmer with bobyqa optimizer."""
    display(Markdown(f'### {label}'))
    print(f'Formula: {formula}')
    m = glmer(formula, data=_to_polars(data), family='binomial')
    _setup_factors(m)
    m.fit(control="glmerControl(optimizer='bobyqa', optCtrl=list(maxfun=500000))")

    # Check singularity
    if is_singular(m):
        print(f'  ⚠  isSingular — between-model variance is on boundary.')
        print(f'     This is expected with n=6 model levels; proceed cautiously.')

    # Fit statistics
    try:
        display(m.result_fit_stats.to_pandas().round(4))
    except Exception:
        print('  [fit stats unavailable]')

    show_fe(m, label, exponentiate=True)
    vc = extract_var_components(m, label)

    try:
        m.anova()
        anova_tbl = m.result_anova.to_pandas()
        p_col = next((c for c in anova_tbl.columns if 'p' in c.lower() and 'npar' not in c.lower()), None)
        if p_col:
            anova_tbl['sig'] = anova_tbl[p_col].map(
                lambda p: '***' if pd.notnull(p) and p < .001
                          else '**' if pd.notnull(p) and p < .01
                          else '*'  if pd.notnull(p) and p < .05
                          else '')
        display(Markdown(f'**{label} — Type-III Wald chi-square tests**'))
        display(anova_tbl.round(4))
    except Exception as e:
        print(f'  [ANOVA: {e}]')

    gc.collect()
    try:
        ro.r('gc(verbose=FALSE)')
    except Exception:
        pass
    return m, vc


def fit_lmm_model(formula, data, label=''):
    """Fit a LMM (Gaussian) via pymer4 lmer with bobyqa optimizer."""
    display(Markdown(f'### {label}'))
    print(f'Formula: {formula}')
    m = lmer(formula, data=_to_polars(data))
    _setup_factors(m)
    m.fit(control="lmerControl(optimizer='bobyqa', optCtrl=list(maxfun=500000))")

    if is_singular(m):
        print(f'  ⚠  isSingular — between-model variance is on boundary.')

    try:
        display(m.result_fit_stats.to_pandas().round(4))
    except Exception:
        pass

    show_fe(m, label)
    vc = extract_var_components(m, label)

    try:
        m.anova()
        anova_tbl = m.result_anova.to_pandas()
        p_col = next((c for c in anova_tbl.columns if 'p' in c.lower() and 'npar' not in c.lower()), None)
        if p_col:
            anova_tbl['sig'] = anova_tbl[p_col].map(
                lambda p: '***' if pd.notnull(p) and p < .001
                          else '**' if pd.notnull(p) and p < .01
                          else '*'  if pd.notnull(p) and p < .05
                          else '')
        display(Markdown(f'**{label} — Type-III F-tests (Satterthwaite)**'))
        display(anova_tbl.round(4))
    except Exception as e:
        print(f'  [ANOVA: {e}]')

    gc.collect()
    try:
        ro.r('gc(verbose=FALSE)')
    except Exception:
        pass
    return m, vc


print('Helpers defined.')


---
## 3. Model 1 — Reading Hallucination GLMM

**Subset:** perceived-source trials only (N = 1,728)  
**Family:** binomial logit  
**Random effect:** `(1 | model)` — between-model intercept variation  
**Fixed effects:** `memory` (SingleTurn vs TrialChain), `order_c`, `rating_cen`  

**Original fixed-effects formula (exp_1_ana4.ipynb):**
> `read_hallucination ~ C(memory) + C(model) + C(order) + rating_cen`

**Sensitivity formula:**
> `read_hallucination ~ memory + order_c + rating_cen + (1 | model)`

**Key comparison:** Does the `memory` effect remain significant when between-model variance is
partitioned out rather than absorbed into 5 fixed dummy contrasts?

In [ ]:
RH_FORMULA  = "read_hallucination ~ memory + order_c + rating_cen + (1 | model)"
rh_cols     = ['read_hallucination', 'memory', 'order_c', 'rating_cen', 'model']
rh_data     = df_per[rh_cols].dropna()

print(f'Reading Hallucination n = {len(rh_data):,}  '
      f'(RH rate = {rh_data["read_hallucination"].mean():.3f})')

m_rh, vc_rh = fit_glmm(RH_FORMULA, rh_data, label='RH — GLMM (1|model)')

---
## 4. Model 2 — Recognition Accuracy GLMM

**Subset:** perceived-source trials only (N = 1,728)  
**Family:** binomial logit  
**Random effect:** `(1 | model)`  
**Fixed effects:** `memory`, `read_hallucination`, `order_c`, `rating_cen`  

**Original formula:**
> `accuracy ~ C(memory) + C(model) + C(read_hallucination) + C(order) + rating_cen`

**Sensitivity formula:**
> `accuracy ~ memory + read_hallucination + order_c + rating_cen + (1 | model)`

In [ ]:
ACC_FORMULA = "accuracy ~ memory + read_hallucination + order_c + rating_cen + (1 | model)"
acc_cols    = ['accuracy', 'memory', 'read_hallucination', 'order_c', 'rating_cen', 'model']
acc_data    = df_per[acc_cols].dropna()

print(f'Accuracy n = {len(acc_data):,}  '
      f'(accuracy rate = {acc_data["accuracy"].mean():.3f})')

m_acc, vc_acc = fit_glmm(ACC_FORMULA, acc_data, label='Accuracy — GLMM (1|model)')

---
## 5. Model 3 — Relatedness Rating LMM

**Subset:** all trials (N = 3,456)  
**Family:** Gaussian identity  
**Random effect:** `(1 | model)`  
**Fixed effects:** `source`, `accuracy`, `memory`, `read_hallucination`, `order_c`  

**Original formula:**
> `rating_cen ~ C(accuracy) + C(source) + C(memory) + C(model) + C(read_hallucination) + C(order)`  
> (OLS with HC3 robust SEs — no random effects)

**Sensitivity formula:**
> `rating_cen ~ source + accuracy + memory + read_hallucination + order_c + (1 | model)`

**Note:** The original analysis used OLS with HC3 robust SEs (statsmodels). This sensitivity analysis
uses REML-estimated LMM, which handles between-model clustering but assumes homoscedastic Gaussian
residuals. The two approaches are not directly comparable but address the same scientific question.

In [ ]:
RR_FORMULA  = "rating_cen ~ source + accuracy + memory + read_hallucination + order_c + (1 | model)"
rr_cols     = ['rating_cen', 'source', 'accuracy', 'memory', 'read_hallucination', 'order_c', 'model']
rr_data     = df[rr_cols].dropna()

print(f'Relatedness rating n = {len(rr_data):,}')

m_rr, vc_rr = fit_lmm_model(RR_FORMULA, rr_data, label='Relatedness Rating — LMM (1|model)')

---
## 6. Variance Component Summary

Between-model ICC quantifies the proportion of total outcome variance attributable to
LLM-architecture identity. A high ICC would indicate that model-level differences dominate
and that fixed-effects inferences about other predictors could be severely confounded;
a low ICC would suggest model identity is a relatively minor source of variation.

In [ ]:
summary_rows = []
for label, vc in [('Reading Hallucination', vc_rh),
                  ('Recognition Accuracy',  vc_acc),
                  ('Relatedness Rating',    vc_rr)]:
    if vc is None:
        summary_rows.append({'Outcome': label, 'σ²_model': 'NA',
                             'σ²_total': 'NA', 'ICC_model': 'NA'})
        continue
    summary_rows.append({
        'Outcome':   label,
        'σ²_model':  round(vc.get('model_var',  np.nan), 4),
        'σ²_total':  round(vc.get('total_var',  np.nan), 4),
        'ICC_model': round(vc.get('icc_model',  np.nan), 4),
    })

icc_tbl = pd.DataFrame(summary_rows)
display(Markdown('**Variance Component Summary — Between-Model ICC**'))
display(icc_tbl)
print()
print('Note: ICC_model = σ²_model / σ²_total (proportion of variance attributable to model identity).')
print('With n=6 model levels, variance estimates are imprecise; treat as indicative only.')

---
## 7. Comparison: Fixed-Effects vs Random-Effects Model

This section summarises the key fixed-effect inferences from the original analysis
alongside the random-effects sensitivity estimates.

In [ ]:
display(Markdown('''
### Inference Replication Checklist

For each primary comparison, report whether the direction and significance of the effect
is consistent between the original fixed-effects model and this random-effects model.

Refer to the coefficient tables in Sections 3–5 above.

| Outcome | Predictor | Original direction | Replicates? |
|---|---|---|---|
| Reading Hallucination | memory (TrialChain) | ? | check Sec 3 |
| Recognition Accuracy  | memory (TrialChain) | ? | check Sec 4 |
| Recognition Accuracy  | read_hallucination  | ? | check Sec 4 |
| Relatedness Rating    | source (imagined)   | ? | check Sec 5 |
| Relatedness Rating    | accuracy (correct)  | ? | check Sec 5 |

*Update the "Replicates?" column after running the notebook.*
'''))

# AIC comparison
print('AIC values (random-effects models):')
for label, m in [('RH',     m_rh),
                 ('Acc',    m_acc),
                 ('Rating', m_rr)]:
    try:
        aic = float(m.result_fit_stats['AIC'][0])
        print(f'  {label}: AIC = {aic:.1f}')
    except Exception:
        print(f'  {label}: AIC unavailable')

---
## 8. Conclusions

### Interpretation guidelines

**If ICC_model is low (< 0.10):** Model identity accounts for a minor fraction of variance.
Fixed-effect inferences about memory format, source, and reading hallucinations are unlikely
to change materially under random-model specification. The fixed-effects model is a reasonable
approximation, and the generalizability limitation is primarily conceptual (specific vs.
population inference) rather than empirical.

**If ICC_model is moderate–high (≥ 0.10):** Model identity is a substantial source of variance.
Fixed-effects inferences may conflate LLM-architecture effects with experimental manipulations.
The random-effects model provides a more appropriate basis for population inference.

**If the model is singular:** The between-model variance estimate is exactly zero or on the
boundary of the parameter space. This can occur when n_groups = 6 is too small to distinguish
between-model variance from sampling noise. In this case, the random-effects model reduces
to the model without a model random effect, and fixed-effects inferences are unchanged.

### Limitation of this analysis

Treating `model` as a random effect assumes the six architectures are drawn from a larger
population of LLMs. With only six levels, the between-model variance estimate has very wide
uncertainty. Random-effects meta-analytic methods (e.g., Hedges & Vevea, 1998) would require
many more model replications to yield precise population-level estimates. This sensitivity
analysis is therefore primarily confirmatory: it checks whether fixed-effect conclusions hold
under a more conservative random-model specification, not whether the population variance
is precisely estimated.